# Semana 9 — RAG con citas · Glaciares
**Proyecto:** Enterprise Brain — Gestión del Conocimiento con Knowledge Graph + GraphRAG · Grupo G8

**Entrada (solo lectura):** carpeta del grupo `proyecto_gc/` → `texto_glaciares/` (Persona 2) y `corpus_glaciares/papers.csv` (Persona 1)
**Salida:** **tu propia carpeta** en Mi unidad (`G8_rag_glaciares/`) → fragmentos y embeddings, respuestas con citas, archivo para RAGAS y un `.zip` para entregar.
El notebook **nunca escribe** en la carpeta del grupo.

| Paso | Tecnología | Para qué |
|---|---|---|
| Material | fragmentos + tablas + resúmenes | ~30 000 piezas de texto, cada una con documento, página y año |
| Embeddings | **bge-m3** (sentence-transformers) | Convierte cada fragmento en un vector de significado. Multilingüe: una pregunta en español encuentra textos en inglés |
| Base vectorial | **Chroma** | Guarda los vectores y busca los más parecidos a la pregunta |
| Palabras clave | **BM25** | Encuentra nombres propios exactos (Palcacocha, Tullparaju) que la búsqueda semántica a veces pierde |
| Generación | **LangChain** + Groq o Gemini | El LLM responde **solo** con los fragmentos y cita cada dato con [n] |
| Verificación | Python | Detecta citas inventadas, oraciones sin cita y cifras que no están en la fuente |

### Antes de ejecutar (una sola vez)
1. En Drive → *Compartido conmigo* → clic derecho en `proyecto_gc` → *Organizar → Agregar acceso directo* → **Mi unidad**
   (funciona aunque solo tengas permiso de lectura). Si no se puede, descarga `papers.csv`, `chunks.jsonl.gz` y `tables.jsonl`
   y súbelos a `Mi unidad/G8_rag_glaciares/entrada/`.
2. Crea en **tu** Mi unidad la carpeta `G8_rag_glaciares/` y sube ahí `rag_glaciares.py`.
3. *Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU*** (los embeddings tardan ~10–15 min con GPU; sin GPU, horas).
4. 🔑 **Secrets** de Colab (con *Acceso al notebook*): `GROQ_API_KEY` y/o `GOOGLE_API_KEY`.
5. *Ejecutar todas*. La primera vez calcula los embeddings y los guarda en tu carpeta; las siguientes los carga en segundos.
6. Al final se crea `entrega_rag_glaciares.zip` en tu carpeta: descárgalo y envíalo a quien administra el Drive del grupo.

Sin clave de LLM el notebook **igual funciona**: busca y muestra los fragmentos con sus citas (modo de respaldo para la demo).

## 0. Carpetas

In [1]:
# ✏️ Solo cambia estas rutas si tus carpetas tienen otro nombre
ENTRADA = "/content/drive/MyDrive/proyecto_gc"        # carpeta del grupo (acceso directo, SOLO LECTURA)
SALIDA  = "/content/drive/MyDrive/G8_rag_glaciares"   # TU carpeta: aquí se guarda todo
ENTRADA_LOCAL = r"G:\Mi unidad\proyecto_gc"           # por si se ejecuta fuera de Colab
SALIDA_LOCAL  = r"G:\Mi unidad\G8_rag_glaciares"

import os, sys
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    ENTRADA, SALIDA = ENTRADA_LOCAL, SALIDA_LOCAL

def primera_existente(*rutas):
    return next((r for r in rutas if os.path.exists(r)), None)

# Entradas: primero la carpeta del grupo; si no está, una copia manual en SALIDA/entrada
COPIA = os.path.join(SALIDA, "entrada")
RUTA_PAPERS = primera_existente(os.path.join(ENTRADA, "corpus_glaciares", "papers.csv"),
                                os.path.join(COPIA, "papers.csv"))
CARPETA_TEXTO = next((c for c in (os.path.join(ENTRADA, "texto_glaciares"), COPIA)
                      if primera_existente(os.path.join(c, "chunks.jsonl.gz"), os.path.join(c, "chunks.jsonl"))), None)

# Salidas: solo en TU carpeta (la del grupo no se toca)
CARPETA_RAG    = SALIDA
CARPETA_CHROMA = "/content/chroma_glaciares" if EN_COLAB else os.path.join(SALIDA, "chroma")
os.makedirs(CARPETA_RAG, exist_ok=True)

# El módulo: en tu carpeta o subido directamente a la sesión de Colab
CARPETA_MODULO = next((c for c in (SALIDA, "/content", os.getcwd())
                       if os.path.exists(os.path.join(c, "rag_glaciares.py"))), None)

print("papers.csv     :", RUTA_PAPERS or "❌ NO ENCONTRADO")
print("texto (chunks) :", CARPETA_TEXTO or "❌ NO ENCONTRADO")
print("módulo         :", CARPETA_MODULO or "❌ NO ENCONTRADO")
print("salidas en     :", CARPETA_RAG)
assert RUTA_PAPERS and CARPETA_TEXTO, ("No encuentro los datos del grupo. Agrega el acceso directo a proyecto_gc en Mi unidad "
                                       "o copia papers.csv, chunks.jsonl.gz y tables.jsonl a " + COPIA)
assert CARPETA_MODULO, "Falta subir rag_glaciares.py a " + SALIDA
if not os.path.exists(os.path.join(CARPETA_TEXTO, "tables.jsonl")):
    print("⚠️ No encontré tables.jsonl: se indexará sin tablas")
sys.path.insert(0, CARPETA_MODULO)

Mounted at /content/drive
papers.csv     : /content/drive/MyDrive/proyecto_gc/corpus_glaciares/papers.csv
texto (chunks) : /content/drive/MyDrive/proyecto_gc/texto_glaciares
módulo         : /content/drive/MyDrive/G8_rag_glaciares
salidas en     : /content/drive/MyDrive/G8_rag_glaciares


## 1. Librerías

In [2]:
%pip install -q chromadb rank_bm25 langchain-core langchain-groq langchain-google-genai sentence-transformers

import importlib, json, time
import numpy as np, pandas as pd
from IPython.display import Markdown, display
import rag_glaciares as R
importlib.reload(R)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO (Entorno de ejecución → T4 GPU)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.9/81.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140

## 2. Claves del LLM (desde Secrets)

In [3]:
def leer_secreto(*nombres):
    for nombre in nombres:
        try:
            from google.colab import userdata
            v = userdata.get(nombre)
            if v:
                print(f"{nombre}: cargada desde Secrets")
                return v
        except Exception:
            pass
        if os.environ.get(nombre):
            print(f"{nombre}: cargada desde variable de entorno")
            return os.environ[nombre]
    print(f"{nombres[0]}: no configurada")
    return None

GROQ_API_KEY = leer_secreto("GROQ_API_KEY")
GOOGLE_API_KEY = leer_secreto("GOOGLE_API_KEY", "GEMINI_API_KEY")

GROQ_API_KEY: cargada desde Secrets
GOOGLE_API_KEY: cargada desde Secrets


## 3. Configuración ✏️

In [4]:
CONFIG = {
    "PROVEEDOR": "groq",          # "groq" o "gemini" (si no hay clave del primero, se usa el otro)
    "MODELO": None,               # None = modelo por defecto del módulo (ver crear_llm)
    "K": 8,                       # fragmentos que lee el LLM por pregunta
    "MAX_POR_DOC": 3,             # máximo de fragmentos de un mismo documento
    "UMBRAL_SIM": 0.35,           # se ajusta en el paso 7
    "RECALCULAR": False,          # True = volver a calcular los embeddings aunque ya estén guardados
    "LOTE_EMB": 32,               # bajar a 16 si la GPU se queda sin memoria
    "PAUSA_EVAL": 2.0,            # segundos entre preguntas (límite gratuito del LLM)
}
R.CONFIG_RAG.update({k: v for k, v in CONFIG.items() if k in R.CONFIG_RAG})

## 4. Material y embeddings 🧮
Une los **fragmentos de texto** de la Persona 2, las **tablas** (ahí están los números: áreas, volúmenes, cantidad de lagunas)
y los **resúmenes** del repositorio. Cada pieza conserva `paper_id`, páginas, año e institución: los mismos datos de procedencia del grafo de Neo4j.

In [5]:
ruta_info = os.path.join(CARPETA_RAG, "embeddings_info.json")
t0 = time.time()
if os.path.exists(ruta_info) and not CONFIG["RECALCULAR"]:
    FRAG, EMB, INFO = R.cargar_material(CARPETA_RAG)
    print(f"Embeddings cargados del Drive ({INFO['n']} fragmentos, {INFO['modelo']}, {INFO['fecha']})")
    EMBEDDER = R.EmbedderBGE()                    # se necesita para convertir las preguntas
else:
    FRAG = R.preparar_fragmentos(CARPETA_TEXTO, RUTA_PAPERS)
    print(f"Material preparado: {len(FRAG)} fragmentos. Calculando embeddings con bge-m3…")
    EMBEDDER = R.EmbedderBGE()
    EMB = EMBEDDER.embed_documents(FRAG.texto_indexar.tolist(), lote=CONFIG["LOTE_EMB"])
    R.guardar_material(FRAG, EMB, CARPETA_RAG, EMBEDDER.nombre)
    print("Guardado en el Drive:", CARPETA_RAG)
print(f"Listo en {(time.time() - t0) / 60:.1f} min · vectores {EMB.shape}")
display(R.resumen_material(FRAG))
display(pd.crosstab(FRAG.fuente, FRAG.idioma, margins=True, margins_name="total"))

Embeddings cargados del Drive (30077 fragmentos, BAAI/bge-m3, 2026-10-18 15:22)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Listo en 1.4 min · vectores (30077, 1024)


,fragmentos,documentos,caracteres_medios
tipo,,,
resumen,398,398,934
tabla,3613,196,1181
texto,26066,373,1039
TOTAL,30077,398,1055


idioma,en,es,total
fuente,,,
ANA,1050,12652,13702
INAIGEM,376,12777,13153
SENAMHI,345,2877,3222
total,1771,28306,30077


## 5. Índice: Chroma (semántico) + BM25 (palabras clave) 🗂️

In [6]:
t0 = time.time()
INDICE = R.IndiceRAG(FRAG, EMB, EMBEDDER, CARPETA_CHROMA)
print(f"Chroma: {INDICE.col.count()} vectores · BM25 listo · {time.time() - t0:.0f} s")

Chroma: 30077 vectores · BM25 listo · 65 s


## 6. ¿Qué encuentra cada buscador? 🔎
Misma pregunta con los tres modos. El **híbrido** fusiona los rankings con *Reciprocal Rank Fusion*:
un fragmento que aparece arriba en los dos buscadores sube al primer lugar.

In [7]:
def ver_busqueda(pregunta, modo="hibrido", filtros=None, k=6):
    r = INDICE.buscar(pregunta, k=k, modo=modo, filtros=filtros)
    r = r.assign(referencia=[R.referencia(f) for f in r.itertuples()],
                 inicio=r.text.str.replace(r"\s+", " ", regex=True).str[:110])
    return r[["n", "tipo", "referencia", "sim", "bm25", "inicio"]]

P = "¿Qué obras de seguridad tiene la laguna Tullparaju?"
for modo in ["semantico", "bm25", "hibrido"]:
    print(f"\n▶ {modo.upper()}")
    with pd.option_context("display.max_colwidth", 90):
        display(ver_busqueda(P, modo))


▶ SEMANTICO


,n,tipo,referencia,sim,bm25,inicio
0,1,texto,"Evaluación de las disponibilidades hídricas en las lagunas de las cuencas: Santa, Nepe...",0.669811,NaN,"Asimismo, en la parte frontal existe infraestructura hidráulica de seguridad, por lo c..."
1,2,texto,"Inspección de Emergencia Laguna Tullparaju - Febrero 2017 — INAIGEM, 2017, pp. 4–6",0.628451,NaN,- Se recomienda instalar una regla milimétrica en la entrada del ducto de la laguna Tu...
2,3,texto,"Inspección de Emergencia Laguna Tullparaju - Febrero 2017 — INAIGEM, 2017, p. 3",0.628243,NaN,Se realizó la inspección de la laguna Tullparaju por encargo de la Dirección de Invest...
3,4,texto,"Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024,...",0.628176,NaN,"a) Descripción general de la laguna Coordenadas: UTM 8957937 N, y 242946 E de la Zona ..."
4,5,texto,"Inspección de Emergencia Laguna Tullparaju - Febrero 2017 — INAIGEM, 2017, p. 4",0.603503,NaN,"Las estructuras de seguridad no presentan daños físicos, el conducto de drenaje se enc..."
5,6,texto,"Inspección de las lagunas: Cuchillacocha, Tullparaju — ANA, 2001, pp. 2–3",0.597377,NaN,"Por medidas de seguridad, se proyectaron y ejecutaron obras hidráulicas de seguridad, ..."



▶ BM25


,n,tipo,referencia,sim,bm25,inicio
0,1,texto,"Inspección de Emergencia Laguna Tullparaju - Febrero 2017 — INAIGEM, 2017, p. 4",NaN,19.695091,- El vaso de la laguna de su flanco lateral derecho e izquierdo no presentan cambios f...
1,2,texto,"Inspección de las lagunas: Cuchillacocha, Tullparaju — ANA, 2001, pp. 2–3",NaN,18.942515,"Por medidas de seguridad, se proyectaron y ejecutaron obras hidráulicas de seguridad, ..."
2,3,texto,"Inspección de Emergencia Laguna Tullparaju - Febrero 2017 — INAIGEM, 2017, p. 4",NaN,18.897397,"Las estructuras de seguridad no presentan daños físicos, el conducto de drenaje se enc..."
3,4,texto,"Informe de la situación de los glaciares y ecosistemas de montaña en el Perú — ANA, 20...",NaN,18.085274,HUARAZ HUAYLAS HUAYLAS HUAYLAS HUAYLAS HUAYLAS BAJO BAJO ALTO ALTO ALTO se determinaro...
4,5,texto,Situación de los Glaciares y Ecosistemas de Montaña en el Perú : Informe Anual 2017 — ...,NaN,17.765535,HUARAZ HUAYLAS HUAYLAS HUAYLAS HUAYLAS HUAYLAS BAJO BAJO ALTO ALTO ALTO se determinaro...
5,6,texto,"Evaluación de las disponibilidades hídricas en las lagunas de las cuencas: Santa, Nepe...",NaN,17.479180,"Asimismo, en la parte frontal existe infraestructura hidráulica de seguridad, por lo c..."



▶ HIBRIDO


,n,tipo,referencia,sim,bm25,inicio
0,1,texto,"Evaluación de las disponibilidades hídricas en las lagunas de las cuencas: Santa, Nepe...",0.669811,17.479180,"Asimismo, en la parte frontal existe infraestructura hidráulica de seguridad, por lo c..."
1,2,texto,"Inspección de las lagunas: Cuchillacocha, Tullparaju — ANA, 2001, pp. 2–3",0.597377,18.942515,"Por medidas de seguridad, se proyectaron y ejecutaron obras hidráulicas de seguridad, ..."
2,3,texto,"Inspección de Emergencia Laguna Tullparaju - Febrero 2017 — INAIGEM, 2017, p. 4",0.603503,18.897397,"Las estructuras de seguridad no presentan daños físicos, el conducto de drenaje se enc..."
3,4,texto,"Inspección de Emergencia Laguna Tullparaju - Febrero 2017 — INAIGEM, 2017, pp. 4–6",0.628451,16.805243,- Se recomienda instalar una regla milimétrica en la entrada del ducto de la laguna Tu...
4,5,texto,Situación de los Glaciares y Ecosistemas de Montaña en el Perú : Informe Anual 2017 — ...,0.573142,17.765535,HUARAZ HUAYLAS HUAYLAS HUAYLAS HUAYLAS HUAYLAS BAJO BAJO ALTO ALTO ALTO se determinaro...
5,6,texto,"Informe de la situación de los glaciares y ecosistemas de montaña en el Perú — ANA, 20...",0.565492,18.085274,HUARAZ HUAYLAS HUAYLAS HUAYLAS HUAYLAS HUAYLAS BAJO BAJO ALTO ALTO ALTO se determinaro...


## 7. Calibrar el umbral "fuera del corpus" 🎚️
Si la pregunta no se parece a nada del corpus, el sistema dice que no sabe **sin llamar al LLM**.
Se compara la similitud máxima de preguntas del tema con preguntas ajenas y se elige un umbral entre ambas.

In [8]:
DEL_TEMA = ["¿Por qué la laguna Palcacocha es peligrosa?", "¿Cuánto retrocedió el glaciar Pastoruri?",
            "¿Qué lagunas tienen riesgo de desborde en Cusco?", "monitoreo de bofedales altoandinos"]
AJENAS = ["¿Quién ganó el Mundial de fútbol 2022?", "receta de ceviche peruano",
          "¿Cuál es el precio del dólar hoy?", "¿Cómo se programa en JavaScript?"]
sims = lambda qs: [round(float(INDICE.buscar(q, k=1, modo="semantico").sim.iloc[0]), 3) for q in qs]
s_tema, s_ajenas = sims(DEL_TEMA), sims(AJENAS)
display(pd.DataFrame({"del tema": DEL_TEMA, "sim": s_tema, "ajena": AJENAS, "sim ": s_ajenas}))
sugerido = round((min(s_tema) + max(s_ajenas)) / 2, 2)
print(f"Mínimo del tema {min(s_tema)} · máximo ajeno {max(s_ajenas)} → umbral sugerido {sugerido}")
if min(s_tema) > max(s_ajenas):
    INDICE.cfg["UMBRAL_SIM"] = sugerido
    print("Umbral aplicado:", sugerido)
else:
    print("Las dos zonas se cruzan: se deja", INDICE.cfg["UMBRAL_SIM"], "y el LLM decide (regla 6 del prompt).")

,del tema,sim,ajena,sim
0,¿Por qué la laguna Palcacocha es peligrosa?,0.661,¿Quién ganó el Mundial de fútbol 2022?,0.402
1,¿Cuánto retrocedió el glaciar Pastoruri?,0.712,receta de ceviche peruano,0.472
2,¿Qué lagunas tienen riesgo de desborde en Cusco?,0.634,¿Cuál es el precio del dólar hoy?,0.497
3,monitoreo de bofedales altoandinos,0.667,¿Cómo se programa en JavaScript?,0.525


Mínimo del tema 0.634 · máximo ajeno 0.525 → umbral sugerido 0.58
Umbral aplicado: 0.58


## 8. El LLM 🤖

In [9]:
LLM = None
claves = {"groq": GROQ_API_KEY, "gemini": GOOGLE_API_KEY}
orden = [CONFIG["PROVEEDOR"]] + [p for p in claves if p != CONFIG["PROVEEDOR"]]
for prov in orden:
    if not claves[prov]:
        continue
    try:
        cand = R.crear_llm(prov, CONFIG["MODELO"] if prov == CONFIG["PROVEEDOR"] else None, claves[prov])
        cand.invoke("Responde solo: ok")
        LLM, PROVEEDOR = cand, prov
        print(f"LLM listo: {prov} · {getattr(cand, 'model_name', None) or getattr(cand, 'model', '')}")
        break
    except Exception as e:
        print(f"⚠️ {prov} no respondió: {str(e)[:200]}")
if LLM is None:
    print("Sin LLM: modo de respaldo (solo fragmentos y citas).")
print("\nInstrucciones que recibe el LLM:\n" + R.INSTRUCCIONES)

LLM listo: groq · openai/gpt-oss-120b

Instrucciones que recibe el LLM:
Eres el asistente de Enterprise Brain Glaciares: respondes preguntas sobre glaciares, lagunas glaciares y riesgo de desborde en el Perú usando documentos del INAIGEM, la ANA y el SENAMHI.

Reglas:
1. Usa SOLO la información de los fragmentos numerados. No uses conocimiento propio.
2. Termina cada oración que contenga un dato con su cita entre corchetes, por ejemplo [2] o [1][4].
3. No inventes cifras, nombres ni fechas. Copia los números tal como aparecen en el fragmento.
4. Si los datos son de años distintos, di el año de cada dato y da prioridad al más reciente. Si los fragmentos se contradicen, dilo.
5. Algunos fragmentos están en inglés: responde siempre en español.
6. Si los fragmentos no permiten responder, responde exactamente: "No encontré esa información en los documentos del corpus." Si solo permiten una respuesta parcial, responde lo que puedas y di qué falta.
7. Sé claro y breve: máximo 200 palabras, si

## 9. Preguntas con citas 🎯
Cada respuesta trae sus fuentes con **enlace al PDF en la página citada** y la ficha del repositorio.
Los avisos ⚠️ salen de la verificación automática.

In [10]:
def preguntar(pregunta, filtros=None, ver_fragmentos=False):
    r = R.responder(pregunta, INDICE, LLM, filtros=filtros)
    display(Markdown(R.a_markdown(r, ver_fragmentos)))
    display(Markdown("---"))
    return r

r1 = preguntar("¿Por qué la laguna Palcacocha es peligrosa?", ver_fragmentos=True)

**Pregunta:** ¿Por qué la laguna Palcacocha es peligrosa?

La laguna Palcococha es considerada peligrosa porque contiene un gran volumen de agua que, unido a la morfología de su microcuenca y a la cercanía de la ciudad de Huaraz, genera un alto riesgo de desborde y aluvión. En 1941 su ruptura liberó aproximadamente 15 Mm³ de agua, lodo y rocas, provocando la muerte de miles de personas y la destrucción de una tercera parte de la ciudad [1]. El retroceso de los glaciares Pucaranra y Palcaraju originó la laguna y, al estar ubicada en la quebrada Cojup que desemboca en el río Paria y luego en el río Santa, cualquier desborde se dirige directamente a Huaraz [3]. En enero 2024 alcanzó su máximo volumen histórico de 17 544 832,63 m³ y una profundidad de 70 m, evidenciando un notable incremento respecto a los 514 800 m³ y 13 m de profundidad registrados en 1974 [3]. Además, la zona presenta glaciares con frentes agrietados que pueden impactar la laguna, y estudios recientes señalan que el sistema de sifonaje propuesto no garantiza seguridad, pues podría represar aún más agua [5]. Todos estos factores combinados hacen que la laguna sea catalogada como peligrosa para la población de Huaraz [4][6].

**Fuentes**
- [1] Evaluación anual de la laguna Palcacocha del año 2024 (seguimiento al comportamiento de glaciares y lagunas a nivel nacional) — ANA, 2024, p. 5 · [abrir en la página](https://repositorio.ana.gob.pe/backend/api/core/bitstreams/a3b97d3b-b37d-473c-a6a0-651bb497a774/content#page=5) · [ficha](https://hdl.handle.net/20.500.12543/5802)
- [3] Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, p. 46 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/58e89821-97de-4197-98d9-cd7900c7f2ec/content#page=46) · [ficha](https://hdl.handle.net/20.500.12748/608)
- [4] Informe de la situación de los glaciares y ecosistemas de montaña en el Perú — ANA, 2019, p. 78 · [abrir en la página](https://repositorio.ana.gob.pe/backend/api/core/bitstreams/a9929c83-280f-4ce6-b6ff-aa05c3ef0c39/content#page=78) · [ficha](https://hdl.handle.net/20.500.12543/4601)
- [5] Reporte hidrológico anual de actividades en la Laguna Palcacocha, 2016 — INAIGEM, 2016, p. 13 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/f4a92b38-5dcf-4aef-9451-22b8883b1585/content#page=13) · [ficha](https://hdl.handle.net/20.500.12748/334)
- [6] Evaluación anual de la laguna Palcacocha del año 2024 (seguimiento al comportamiento de glaciares y lagunas a nivel nacional) — ANA, 2024, p. 5 · [abrir en la página](https://repositorio.ana.gob.pe/backend/api/core/bitstreams/a3b97d3b-b37d-473c-a6a0-651bb497a774/content#page=5) · [ficha](https://hdl.handle.net/20.500.12543/5802)

<sub>1.2 s · similitud máxima 0.654</sub>

**Fragmentos recuperados**
- [1] *texto* · Evaluación anual de la laguna Palcacocha del año 2024 (seguimiento al comportamiento de glaciares y lagunas a nivel nacional) — ANA, 2024, p. 5 · sim 0.65
- [2] *texto* · Evaluación anual de la laguna Palcacocha del año 2024 (seguimiento al comportamiento de glaciares y lagunas a nivel nacional) — ANA, 2024, p. 6 · sim 0.65
- [3] *texto* · Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, p. 46 · sim 0.65
- [4] *texto* · Informe de la situación de los glaciares y ecosistemas de montaña en el Perú — ANA, 2019, p. 78 · sim 0.59
- [5] *texto* · Reporte hidrológico anual de actividades en la Laguna Palcacocha, 2016 — INAIGEM, 2016, p. 13 · sim 0.61
- [6] *texto* · Evaluación anual de la laguna Palcacocha del año 2024 (seguimiento al comportamiento de glaciares y lagunas a nivel nacional) — ANA, 2024, p. 5 · sim 0.61
- [7] *texto* · Informe técnico de la situación actual de la Laguna Glaciar Palcacocha - Enero — INAIGEM, 2016, p. 2 · sim 0.59
- [8] *texto* · Revista Científica de Glaciares y Ecosistemas de Montaña 5 — INAIGEM, 2018, pp. 67–68 · sim 0.60

---

In [11]:
# Pregunta cuya respuesta está en documentos en inglés: bge-m3 los encuentra y el LLM responde en español
r2 = preguntar("¿Qué factores explican el balance de masa y energía de los glaciares peruanos?")

**Pregunta:** ¿Qué factores explican el balance de masa y energía de los glaciares peruanos?

Los factores que explican el balance de masa y el balance de energía de los glaciares peruanos son:

* **Variables atmosféricas**: temperatura del aire, humedad relativa, velocidad del viento, presión atmosférica y precipitación, que influyen tanto en la acumulación de nieve como en la ablación por fusión [1].  
* **Radiación**: radiación solar de onda corta incidente (medida directamente) y radiación neta de onda larga, que aportan energía positiva y negativa al glaciar respectivamente [1][8].  
* **Flujos de calor**: calor sensible y latente turbulento, y flujo de calor subsuperficial, que modifican la energía disponible para la fusión [1][6].  
* **Albedo**: la presencia de nieve fresca aumenta el albedo y reduce la absorción de radiación, mientras que la exposición de hielo o sedimentos oscuros (crioconitas) disminuye el albedo y acelera el derretimiento [4][5][7].  
* **Precipitación**: cantidad y forma (nieve) determinan la acumulación de masa; la variabilidad de la precipitación es el principal impulsor de las fluctuaciones del balance de masa en la Cordillera Blanca [2][8].  
* **Elevación y gradiente altitudinal**: la distribución del balance de masa cambia con la altitud; zonas por encima de la línea de equilibrio acumulan masa y por debajo la pierden [3][5][7].  
* **Factores orográficos y geológicos**: diferencias entre cordilleras volcánicas y no volcánicas modifican los patrones altitudinales del balance de masa [3].  
* **Cobertura de nieve y hielo**: la topografía heterogénea y la acumulación de nieve en capas influyen en la energía disponible para la ablación [4].  

Estos elementos interactúan para determinar si un glaciar gana o pierde masa y cómo se distribuye su energía superficial.

**Fuentes**
- [1] Revista de glaciares y ecosistemas de montaña 7 — INAIGEM, 2024, p. 63 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/f1e9b8ad-472c-45e7-9fed-1dbd3c9e0f77/content#page=63) · [ficha](http://hdl.handle.net/20.500.12748/531)
- [2] Glacier mass balance variability in the Cordillera Blanca, Peru and its relationship with climate and the large-scale circulation — ANA, 2008, resumen · [abrir en la página](https://hdl.handle.net/20.500.12543/3334) · [ficha](https://hdl.handle.net/20.500.12543/3334)
- [3] Libro de resumenes Simposio MONFU 2025 — INAIGEM, 2025, pp. 65–66 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/de263cb8-7bba-4840-9619-a2cf401b861e/content#page=65) · [ficha](https://hdl.handle.net/20.500.12748/651)
- [4] Iniciación de un monitoreo del balance de masa en el glaciar Suyuparina, Cordillera Vilcanota, Perú — ANA, 2015, pp. 11–12 · [abrir en la página](https://repositorio.ana.gob.pe/backend/api/core/bitstreams/d52e9a8e-63dc-44e2-b4c4-7d5c4bfa0c9b/content#page=11) · [ficha](https://hdl.handle.net/20.500.12543/3185)
- [5] Estudios de glaciares por la Autoridad Nacional del Agua - Perú, período 2022-2023 (Meta: seguimiento al comportamiento de glaciares y lagunas a nivel nacional) — ANA, 2023, pp. 87–88 · [abrir en la página](https://repositorio.ana.gob.pe/backend/api/core/bitstreams/3708734d-e5ab-4c00-8d6f-6a11690d4c16/content#page=87) · [ficha](https://hdl.handle.net/20.500.12543/5684)
- [6] Estudio científico sobre paleoclima, volumen 3 Nº2 (1997) — SENAMHI, 1997, pp. 11–12 · [abrir en la página](https://repositorio.senamhi.gob.pe/rest/bitstreams/aeb3160e-c07e-4767-af71-4fd95f7db12a/retrieve#page=11) · [ficha](https://hdl.handle.net/20.500.12542/4848)
- [7] Estudios de glaciares por la Autoridad Nacional del Agua - Perú, período 2022-2023 (Meta: seguimiento al comportamiento de glaciares y lagunas a nivel nacional) — ANA, 2023, pp. 71–72 · [abrir en la página](https://repositorio.ana.gob.pe/backend/api/core/bitstreams/3708734d-e5ab-4c00-8d6f-6a11690d4c16/content#page=71) · [ficha](https://hdl.handle.net/20.500.12543/5684)
- [8] Variaciones glaciares y disponibilidad de agua en la cordillera Blanca del Perú desde hace 40 año — SENAMHI, 2010, p. 5 · [abrir en la página](https://repositorio.senamhi.gob.pe/rest/bitstreams/771b92d4-67e7-4799-b7fa-060da8693420/retrieve#page=5) · [ficha](https://hdl.handle.net/20.500.12542/2017)

<sub>1.4 s · similitud máxima 0.671</sub>

---

In [12]:
# Filtros: la misma pregunta solo con documentos recientes del INAIGEM vs. documentos antiguos de la ANA
P = "¿Cuál es la situación de la laguna Parón?"
r3 = preguntar(P, filtros={"fuente": ["INAIGEM"], "anio_min": 2018})
r4 = preguntar(P, filtros={"fuente": ["ANA"], "anio_max": 2010})

**Pregunta:** ¿Cuál es la situación de la laguna Parón?

La laguna Parón presenta una alta vulnerabilidad a desbordes y aluviones. Según el informe de 2018, la peligrosidad alrededor de la laguna varía de moderada a muy alta, influida por una lengua glaciar de gran dimensión en la parte baja de la laguna Artesoncocha Alta y por el gran volumen de agua almacenado, superior a 79 Mm³ (estudio batimétrico 2006) [1][2]. En la evaluación de susceptibilidad por movimientos en masa se identifican niveles de peligro alto y muy alto en las zonas bajas de la subcuenca, especialmente en el entorno de la laguna [3]. El estudio nacional de 2024 indica que la laguna posee taludes laterales pronunciados (pendientes de hasta 35,6 °) y un volumen de 73 348 527 m³ de agua en abril 2024, lo que incrementa su susceptibilidad a procesos GLOF [6]. El sistema de regulación de inundaciones no está siendo controlado técnicamente y el túnel de 1 157 m construido en 1972 no garantiza seguridad [4][6]. Poblaciones aguas abajo (Cono, Cumpayhuara, Caraz y otros asentamientos) están expuestas: 545 viviendas y 22 303 habitantes, 7 738 viviendas y 27 estructuras de importancia según la evaluación de riesgo de 2020 [5]. En conjunto, la laguna Parón se considera una de las lagunas glaciares más grandes y peligrosas de la Cordillera Blanca, con riesgo elevado de desbordamiento violento y aluviones.

**Fuentes**
- [1] Situación de los Glaciares y Ecosistemas de Montaña en el Perú : Informe Anual 2018 — INAIGEM, 2019, pp. 49–51 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/0ea0fc8f-e601-4a8c-87c8-d2ba3a80e26c/content#page=49) · [ficha](https://hdl.handle.net/20.500.12748/139)
- [2] Situación de los Glaciares y Ecosistemas de Montaña en el Perú : Informe Anual 2018 — INAIGEM, 2019, p. 129 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/0ea0fc8f-e601-4a8c-87c8-d2ba3a80e26c/content#page=129) · [ficha](https://hdl.handle.net/20.500.12748/139)
- [3] Situación de los Glaciares y Ecosistemas de Montaña en el Perú : Informe Anual 2018 — INAIGEM, 2019, p. 49 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/0ea0fc8f-e601-4a8c-87c8-d2ba3a80e26c/content#page=49) · [ficha](https://hdl.handle.net/20.500.12748/139)
- [4] Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, p. 52 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/58e89821-97de-4197-98d9-cd7900c7f2ec/content#page=52) · [ficha](https://hdl.handle.net/20.500.12748/608)
- [5] Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, pp. 52–55 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/58e89821-97de-4197-98d9-cd7900c7f2ec/content#page=52) · [ficha](https://hdl.handle.net/20.500.12748/608)
- [6] Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, p. 52 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/58e89821-97de-4197-98d9-cd7900c7f2ec/content#page=52) · [ficha](https://hdl.handle.net/20.500.12748/608)

> ⚠️ Cifras que no aparecen tal cual en los fragmentos citados: 73 348 527

<sub>17.4 s · similitud máxima 0.67</sub>

---

**Pregunta:** ¿Cuál es la situación de la laguna Parón?

La laguna Parón está rodeada por lagunas de dimensiones reducidas y de bajo volumen, y los glaciares que la alimentan son en su mayoría pequeños y de pendiente moderada[1]. En la margen derecha y parte posterior de la laguna se observan embalsamientos que han roto su dique natural; el material desprendido se ha acumulado cerca del borde formando conos aluviales, aunque sin afectar la estabilidad de la cubeta[1]. Además, se registran caídas de bloques de roca hacia la cubeta, procedentes del glaciar Hatunraju, y la zona presenta sistemas de fracturamiento que pueden producir nuevos desprendimientos con sismos de intensidad regular[1]. Por otro lado, los glaciares de la zona muestran una constante recesión hacia niveles más altos y una disminución del espesor de la capa de hielo, aunque en menor magnitud que en otros glaciares debido a su ubicación cercana a la zona de equilibrio[3]. 

En resumen, la laguna Parón presenta un dique natural parcialmente comprometido por embalsamientos, riesgo de caída de bloques rocosos y una tendencia a la reducción de los glaciares que la alimentan.

**Fuentes**
- [1] Estudio de la cuenca Alta de Parón — ANA, 1986, p. 19 · [abrir en la página](https://repositorio.ana.gob.pe/backend/api/core/bitstreams/d87fb076-3d0b-4eb6-aaf5-0aac55209770/content#page=19) · [ficha](https://hdl.handle.net/20.500.12543/1661)
- [3] Estudio de la cuenca Alta de Parón — ANA, 1986, pp. 19–20 · [abrir en la página](https://repositorio.ana.gob.pe/backend/api/core/bitstreams/d87fb076-3d0b-4eb6-aaf5-0aac55209770/content#page=19) · [ficha](https://hdl.handle.net/20.500.12543/1661)

<sub>31.2 s · similitud máxima 0.583</sub>

---

In [13]:
# Fuera del corpus: no inventa
r5 = preguntar("¿Quién ganó el Mundial de fútbol 2022?")

**Pregunta:** ¿Quién ganó el Mundial de fútbol 2022?

No encontré esa información en los documentos del corpus.


> ⚠️ La pregunta no se parece a ningún fragmento del corpus.

<sub>0.1 s · similitud máxima 0.402</sub>

---

## 10. Dónde se queda corto el RAG (y por qué hace falta GraphRAG) 🧩
El RAG solo lee **8 fragmentos sueltos**. Para contar, comparar o encadenar entidades de muchos documentos
necesita el grafo: eso es la semana 10.

In [14]:
r6 = preguntar("¿Cuántas lagunas con riesgo muy alto de desborde hay en cada departamento y cuál tiene más?")
r7 = preguntar("¿Qué centros poblados están aguas abajo de lagunas con riesgo de desborde y qué glaciares alimentan esas lagunas?")

**Pregunta:** ¿Cuántas lagunas con riesgo muy alto de desborde hay en cada departamento y cuál tiene más?

En el estudio nacional de 2024 se identificaron **58 lagunas glaciares con riesgo «muy alto»** a nivel del país. De esas, **Áncash concentra 26 lagunas**, **Cusco 13 lagunas** y **Lima 6 lagunas** con riesgo muy alto [4].  

Los demás departamentos que presentan lagunas con riesgo muy alto no tienen su número especificado en los fragmentos disponibles; solo se indica que son seis departamentos en total, pero no se desglosa cuántas lagunas corresponde a cada uno [1].  

Por lo tanto, el departamento con mayor número de lagunas de riesgo muy alto es **Áncash** con 26 lagunas.  

*Falta información detallada del número de lagunas muy altas para los otros departamentos que aparecen en la lista.*

**Fuentes**
- [1] Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, p. 66 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/58e89821-97de-4197-98d9-cd7900c7f2ec/content#page=66) · [ficha](https://hdl.handle.net/20.500.12748/608)
- [4] RESUMEN PARA TOMADORES DE DECISIÓN: EVALUACIÓN NACIONAL DE LAGUNAS GLACIARES CON RIESGO DE DESBORDE — INAIGEM, 2024, pp. 11–12 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/6c7ad585-5c01-44c1-b9e5-ad08f93d9b2c/content#page=11) · [ficha](https://hdl.handle.net/20.500.12748/605)

> ⚠️ 2 oración(es) con datos sin cita.

<sub>26.1 s · similitud máxima 0.755</sub>

---

**Pregunta:** ¿Qué centros poblados están aguas abajo de lagunas con riesgo de desborde y qué glaciares alimentan esas lagunas?

- **Laguna Upiscocha** – alimentada por el nevado **Ausangate**; aguas abajo se encuentran los poblados de **Yanama, Ocongate y Ccarhuayo**[1].  
- **Laguna Vizcachani** – se menciona que comparte exposición al riesgo con la laguna Sorapata; los poblados situados aguas abajo de la laguna de **Sorapata** son **Hanccoccala, Koriwara, Lusuni, Hornopampa, Chacapampa y Potoni**[2]. No se indica el glaciar que alimenta Vizcachani.  
- **Laguna Lazo Huntay** – aguas abajo están la ciudad de **Huancayo** y los poblados de **Acopalca, Chamiseria, Pañaspampa y Vilcacoto**[3]. No se menciona el glaciar que alimenta la laguna.  
- **Laguna de Parón** – los poblados expuestos son **Cono, Cumpayhuara (Santa Rosa) y Caraz**[4]. No se indica el glaciar alimentador.  
- **Laguna Sorapata** – los centros poblados expuestos son **Sina, Potoni y Koriwara** (distrito de Sina)[7]. No se menciona el glaciar que alimenta la laguna.  
- **Laguna Llaca** – alimentada por el **glaciar Llaca 1**; aguas abajo se encuentran los barrios de **Palmira Alta, Palmira Baja, Acovichay, El Milagro y Cascapampa**, así como otros centros poblados a lo largo del río Casca[8].  

**Observación:** En varios fragmentos (Vizcachani, Lazo Huntay, Parón, Sorapata) se describen los poblados downstream pero no se especifica el glaciar que alimenta esas lagunas; por lo tanto, esa información no está disponible en el corpus proporcionado.

**Fuentes**
- [1] RESUMEN PARA TOMADORES DE DECISIÓN: EVALUACIÓN NACIONAL DE LAGUNAS GLACIARES CON RIESGO DE DESBORDE — INAIGEM, 2024, pp. 8–11 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/6c7ad585-5c01-44c1-b9e5-ad08f93d9b2c/content#page=8) · [ficha](https://hdl.handle.net/20.500.12748/605)
- [2] Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, p. 61 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/58e89821-97de-4197-98d9-cd7900c7f2ec/content#page=61) · [ficha](https://hdl.handle.net/20.500.12748/608)
- [3] Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, pp. 63–64 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/58e89821-97de-4197-98d9-cd7900c7f2ec/content#page=63) · [ficha](https://hdl.handle.net/20.500.12748/608)
- [4] Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, pp. 52–55 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/58e89821-97de-4197-98d9-cd7900c7f2ec/content#page=52) · [ficha](https://hdl.handle.net/20.500.12748/608)
- [7] Informe de la Situación de los Glaciares y Ecosistemas de Montaña en el Perú 2022 — INAIGEM, 2025, p. 12 · [abrir en la página](https://repositorio.inaigem.gob.pe/server/api/core/bitstreams/aca384fc-5c51-4deb-8393-726092e7e2cd/content#page=12) · [ficha](https://hdl.handle.net/20.500.12748/667)
- [8] Evaluación del peligro de aluvión por desborde de laguna glaciar en la subcuenca Casca — INAIGEM, 2026, resumen · [abrir en la página](https://hdl.handle.net/20.500.12748/668) · [ficha](https://hdl.handle.net/20.500.12748/668)

<sub>25.2 s · similitud máxima 0.649</sub>

---

## 11. Evaluación con el banco de preguntas 📋
Las 10 preguntas de competencia (✏️ **reemplazar por el texto exacto de la ficha de la semana 5**) y preguntas de demo.
Se guarda un CSV para revisar y un JSONL con pregunta, respuesta y contextos: la entrada de **RAGAS** en la semana 12.

In [15]:
BANCO = [
    {"id": "P1", "tipo": "competencia", "pregunta": "¿Cuántas lagunas de origen glaciar se formaron en la Cordillera Blanca según el inventario de 2023?"},
    {"id": "P2", "tipo": "competencia", "pregunta": "¿Qué lagunas glaciares de Áncash tienen riesgo muy alto de desborde?"},
    {"id": "P3", "tipo": "competencia", "pregunta": "¿Qué centros poblados están aguas abajo de lagunas con riesgo alto y qué estudios los evaluaron?"},
    {"id": "P4", "tipo": "competencia", "pregunta": "¿Qué lagunas tuvieron eventos de desborde, de qué glaciar provienen y qué poblados afectaron?"},
    {"id": "P5", "tipo": "competencia", "pregunta": "	¿Qué glaciares con retroceso medido (balance de masa) alimentan lagunas que hoy están en riesgo?"},
    {"id": "P6", "tipo": "competencia", "pregunta": "¿Qué métodos de monitoreo se aplicaron a lagunas en riesgo y qué institución realizó cada estudio?"},
    {"id": "P7", "tipo": "competencia", "pregunta": "	¿En qué subcuencas coinciden lagunas en riesgo y contaminación por drenaje ácido de roca?"},
    {"id": "P8", "tipo": "competencia", "pregunta": "¿Qué bofedales están en subcuencas con glaciares en retroceso y qué servicios ecosistémicos brindan?"},
    {"id": "P9", "tipo": "competencia", "pregunta": "¿Cómo cambió la superficie glaciar de una cordillera entre los inventarios de 2018 y 2023?"},
    {"id": "P10", "tipo": "competencia", "pregunta": "¿Qué lagunas en riesgo cuentan con obras de seguridad, en qué estado están y qué poblados protegen?"},
    {"id": "D1", "tipo": "descriptiva", "pregunta": "¿Por qué la laguna Palcacocha es peligrosa?"},
    {"id": "D2", "tipo": "multilingüe", "pregunta": "¿Qué factores explican el balance de masa y energía de los glaciares peruanos?"},
    {"id": "D3", "tipo": "histórica", "pregunta": "¿Qué ocurrió en el aluvión de Huaraz de 1941?"},
    {"id": "D4", "tipo": "conteo", "pregunta": "¿Cuántas lagunas con riesgo muy alto de desborde hay en cada departamento?"},
    {"id": "D5", "tipo": "fuera del corpus", "pregunta": "¿Quién ganó el Mundial de fútbol 2022?"},
]
EVAL, RAGAS = R.evaluar(BANCO, INDICE, LLM, pausa=CONFIG["PAUSA_EVAL"] if LLM else 0)
EVAL.to_csv(os.path.join(CARPETA_RAG, "respuestas_rag.csv"), index=False, encoding="utf-8")
with open(os.path.join(CARPETA_RAG, "ragas_semana9.jsonl"), "w", encoding="utf-8") as f:
    for x in RAGAS:
        f.write(json.dumps(x, ensure_ascii=False) + "\n")
with pd.option_context("display.max_colwidth", 80):
    display(EVAL[["id", "tipo", "responde", "n_citas", "docs_citados", "instituciones_citadas", "avisos", "segundos"]])
c = EVAL[EVAL.tipo == "competencia"]
print(f"Competencia: responde {c.responde.sum()}/{len(c)} · con citas {(c.n_citas > 0).sum()}/{len(c)} · "
      f"citas promedio {c.n_citas.mean():.1f} · sin citas inventadas {(c.citas_invalidas == 0).sum()}/{len(c)}")

,id,tipo,responde,n_citas,docs_citados,instituciones_citadas,avisos,segundos
0,P1,competencia,True,1,1,INAIGEM,1 oración(es) con datos sin cita.,28.0
1,P2,competencia,True,1,1,INAIGEM,,20.8
2,P3,competencia,True,4,2,INAIGEM,,24.5
3,P4,competencia,True,5,5,INAIGEM,,22.8
4,P5,competencia,False,0,0,,,18.8
5,P6,competencia,True,2,2,"ANA, INAIGEM",1 oración(es) con datos sin cita.,18.4
6,P7,competencia,True,3,2,INAIGEM,,22.3
7,P8,competencia,True,3,3,INAIGEM,,20.1
8,P9,competencia,True,3,2,INAIGEM,1 oración(es) con datos sin cita.,28.2
9,P10,competencia,True,5,4,INAIGEM,,13.5


Competencia: responde 9/10 · con citas 9/10 · citas promedio 2.7 · sin citas inventadas 10/10


In [16]:
# Leer una respuesta completa del banco
fila = EVAL.set_index("id").loc["P2"]
display(Markdown(f"**{fila.pregunta}**\n\n{fila.respuesta}\n\n<sub>{fila.fuentes}</sub>"))

**¿Qué lagunas glaciares de Áncash tienen riesgo muy alto de desborde?**

Las lagunas glaciares de Áncash que se ubican en nivel **muy alto** de riesgo de desborde son:

- **Laguna Palcacocha** (provincia de Huaraz)  
- **Laguna Tullparaju** (provincia de Huaraz)  
- **Laguna Cuchillacocha** (provincia de Huaraz)  
- **Laguna Llaca** (provincia de Huaraz)  

[1]

<sub>Evaluación nacional de lagunas glaciares con riesgo de desborde 2024. — INAIGEM, 2024, pp. 68–69</sub>

## 12. El mismo buscador como *retriever* de LangChain 🔗
Es la pieza que la semana 10 combina con Neo4j (GraphRAG). Cada documento trae `paper_id` y páginas,
igual que las relaciones del grafo, así que se puede ir del hecho del grafo al texto que lo respalda.

In [17]:
RETRIEVER = R.crear_retriever(INDICE, k=4)
for d in RETRIEVER.invoke("laguna Palcacocha dique"):
    print(d.metadata["paper_id"], d.metadata["paginas"], "·", d.metadata["titulo"][:70])

20.500.12748/608 p. 46 · Evaluación nacional de lagunas glaciares con riesgo de desborde 2024.
20.500.12748/288 pp. 3–7 · Informe técnico de la situación actual de la Laguna Glaciar Palcacocha
20.500.12543/5802 p. 5 · Evaluación anual de la laguna Palcacocha del año 2024 (seguimiento al 
20.500.12543/2423 p. 3 · Evolución de la laguna glaciar Palcacocha


## 13. Reporte

In [18]:
c = EVAL[EVAL.tipo == "competencia"]
lin = ["# Reporte RAG con citas · semana 9", "",
       f"- Fragmentos indexados: {len(FRAG)} ({', '.join(f'{k}: {v}' for k, v in FRAG.tipo.value_counts().items())})",
       f"- Documentos: {FRAG.paper_id.nunique()} · embeddings: bge-m3 ({EMB.shape[1]} dimensiones) · base vectorial: Chroma + BM25",
       f"- LLM: {PROVEEDOR if LLM else 'sin LLM (respaldo)'} · K = {INDICE.cfg['K']} · umbral = {INDICE.cfg['UMBRAL_SIM']}",
       f"- Preguntas de competencia respondidas: {c.responde.sum()}/{len(c)} · con citas: {(c.n_citas > 0).sum()}/{len(c)}",
       "", "| id | responde | citas | instituciones | avisos |", "|---|---|---|---|---|"]
lin += [f"| {r.id} | {'✅' if r.responde else '❌'} | {r.n_citas} | {r.instituciones_citadas} | {str(r.avisos).replace('|', ';')} |"
        for r in EVAL.itertuples()]
with open(os.path.join(CARPETA_RAG, "reporte_rag.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(lin))
print("\n".join(lin))
import shutil, zipfile
ruta_zip = os.path.join(CARPETA_RAG, "entrega_rag_glaciares.zip")
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for n in ["rag_glaciares.py", "reporte_rag.md", "respuestas_rag.csv", "ragas_semana9.jsonl",
              "embeddings_info.json", "fragmentos_rag.parquet", "embeddings.npy"]:
        origen = os.path.join(CARPETA_MODULO if n == "rag_glaciares.py" else CARPETA_RAG, n)
        if os.path.exists(origen):
            z.write(origen, arcname=f"rag_glaciares/{n}")
print(f"\n📦 Para entregar: {ruta_zip} ({os.path.getsize(ruta_zip) / 1e6:.0f} MB)")
print("   Descárgalo y envíalo a quien administra el Drive: lo descomprime dentro de proyecto_gc/.")
print("   Falta agregar este notebook ya ejecutado (Archivo → Descargar → .ipynb).")

# Reporte RAG con citas · semana 9

- Fragmentos indexados: 30077 (texto: 26066, tabla: 3613, resumen: 398)
- Documentos: 398 · embeddings: bge-m3 (1024 dimensiones) · base vectorial: Chroma + BM25
- LLM: groq · K = 8 · umbral = 0.58
- Preguntas de competencia respondidas: 9/10 · con citas: 9/10

| id | responde | citas | instituciones | avisos |
|---|---|---|---|---|
| P1 | ✅ | 1 | INAIGEM | 1 oración(es) con datos sin cita. |
| P2 | ✅ | 1 | INAIGEM |  |
| P3 | ✅ | 4 | INAIGEM |  |
| P4 | ✅ | 5 | INAIGEM |  |
| P5 | ❌ | 0 |  |  |
| P6 | ✅ | 2 | ANA, INAIGEM | 1 oración(es) con datos sin cita. |
| P7 | ✅ | 3 | INAIGEM |  |
| P8 | ✅ | 3 | INAIGEM |  |
| P9 | ✅ | 3 | INAIGEM | 1 oración(es) con datos sin cita. |
| P10 | ✅ | 5 | INAIGEM |  |
| D1 | ✅ | 3 | ANA, INAIGEM |  |
| D2 | ✅ | 7 | ANA, INAIGEM, SENAMHI |  |
| D3 | ✅ | 6 | ANA, INAIGEM |  |
| D4 | ✅ | 3 | INAIGEM |  |
| D5 | ❌ | 0 |  | La pregunta no se parece a ningún fragmento del corpus. |

📦 Para entregar: /content/drive/MyDriv

## 14. Guion para la demo (≈ 3 min) 🎤
1. **Qué es:** "El RAG busca en los 398 documentos los 8 fragmentos más relevantes y el LLM responde solo con ellos, citando documento y página."
2. **Pregunta descriptiva** (paso 9, Palcacocha): mostrar la respuesta, hacer clic en una cita → se abre el PDF en la página.
3. **Multilingüe** (factores del balance de masa): la fuente está en inglés y la respuesta en español.
4. **Filtros por fecha** (Parón): INAIGEM reciente vs. ANA antigua → el dato cambia con los años y el sistema dice de qué año es cada uno.
5. **No inventa** (Mundial 2022) y la **verificación** marca citas o cifras dudosas.
6. **Límite** (conteo por departamento): el RAG solo ve fragmentos sueltos → "por eso la próxima semana lo combinamos con el grafo: GraphRAG".

**Preguntas probables del jurado**
- *¿Por qué bge-m3?* Multilingüe (≈ 1 800 fragmentos en inglés), textos largos y gratuito.
- *¿Por qué híbrido?* La búsqueda semántica entiende sinónimos; BM25 no pierde nombres propios de lagunas y glaciares.
- *¿Cómo evitan que invente?* Prompt con reglas, umbral de similitud y verificación automática de citas y cifras.
- *¿Por qué indexan tablas?* Los números (áreas, volúmenes, cantidad de lagunas) están en tablas, no en el texto corrido.